# Reporting Analysis — Gaming Compliance & Risk Intelligence Platform

Connects to the platform's Snowflake **reporting layer** and charts the results of the
executed build (validated 2026-07-02: 18/18 setup + 21/21 reconciliation/DQ). This notebook
is the *visual analysis* companion to the SQL platform — it queries the `REPORTING.VW_*`
views and renders the same story the analyst version tells with charts.

**Synthetic data only.** No real customers or market. **No credentials appear in this file.**

### How to run
1. Build the platform in Snowflake (see [`../docs/deployment_guide.md`](../docs/deployment_guide.md)).
2. `pip install -r ../requirements.txt`.
3. Provide your Snowflake connection **without hard-coding secrets** — either a
   `connections.toml` entry named `gaming_compliance`, or the `SNOWFLAKE_*` environment
   variables used below (SSO / browser auth recommended).
4. Run all cells, then **commit the notebook with its outputs** so the charts render on GitHub.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector

# No credentials in this file. Use a connections.toml entry (recommended) or SNOWFLAKE_* env vars.
conn_name = os.environ.get("SNOWFLAKE_CONNECTION", "gaming_compliance")
try:
    conn = snowflake.connector.connect(connection_name=conn_name)
except Exception:
    conn = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        authenticator=os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser"),
        role=os.environ.get("SNOWFLAKE_ROLE", "BI_REPORTING"),
        warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE", "WH_REPORTING"),
        database="GAMING_COMPLIANCE_DB",
    )

def q(sql):
    cur = conn.cursor()
    try:
        cur.execute(sql)
        return cur.fetch_pandas_all()
    finally:
        cur.close()

print(q("SELECT CURRENT_ROLE() AS ROLE, CURRENT_WAREHOUSE() AS WH, CURRENT_DATABASE() AS DB"))

## 1. Executive KPI snapshot
One-row view, transposed to a legible metric → value list.

In [ ]:
kpi = q("SELECT * FROM REPORTING.VW_EXECUTIVE_OVERVIEW").T
kpi.columns = ["value"]
kpi

## 2. AML alerts by typology
All 11 rule typologies and how many alerts each raised.

In [ ]:
typ = q("SELECT RULE_CODE, RULE_NAME, ALERTS, ESCALATED FROM REPORTING.VW_ALERT_TYPOLOGY_BREAKDOWN ORDER BY ALERTS")
ax = typ.plot.barh(x="RULE_CODE", y="ALERTS", legend=False, figsize=(8,5), color="#2563eb")
ax.set_title("AML alerts by typology"); ax.set_xlabel("alerts"); ax.set_ylabel("rule")
plt.tight_layout(); plt.show()
typ

## 3. Monthly compliance trends
Alerts, escalations, and STR cases over the synthetic time window.

In [ ]:
tr = q("SELECT YEAR_MONTH, ALERTS, ESCALATED_ALERTS, STR_CASES FROM REPORTING.VW_MONTHLY_COMPLIANCE_TRENDS ORDER BY YEAR_MONTH")
ax = tr.plot(x="YEAR_MONTH", y=["ALERTS","ESCALATED_ALERTS","STR_CASES"], figsize=(11,4))
ax.set_title("Monthly alerts, escalations, and STR cases"); ax.set_xlabel("month"); ax.set_ylabel("count")
plt.tight_layout(); plt.show()

## 4. Market / GGR performance
Monthly wagers and gross gaming revenue (synthetic, illustrative).

In [ ]:
mkt = q("SELECT YEAR_MONTH, TOTAL_WAGERS, TOTAL_GGR FROM REPORTING.VW_MARKET_PERFORMANCE ORDER BY YEAR_MONTH")
fig, ax1 = plt.subplots(figsize=(11,4))
ax1.plot(mkt["YEAR_MONTH"], mkt["TOTAL_WAGERS"], color="#94a3b8", label="wagers")
ax2 = ax1.twinx()
ax2.plot(mkt["YEAR_MONTH"], mkt["TOTAL_GGR"], color="#16a34a", label="GGR")
ax1.set_title("Wagers vs GGR by month"); ax1.set_xlabel("month")
ax1.set_ylabel("wagers"); ax2.set_ylabel("GGR")
ax1.tick_params(axis='x', labelrotation=90)
plt.tight_layout(); plt.show()

## 5. STR SLA performance by priority
Case volume and SLA breaches per priority tier.

In [ ]:
sla = q("SELECT CASE_PRIORITY, CASES, SLA_BREACHES, SLA_COMPLIANCE_PCT FROM REPORTING.VW_SLA_PERFORMANCE ORDER BY SLA_TARGET_DAYS")
ax = sla.plot.bar(x="CASE_PRIORITY", y=["CASES","SLA_BREACHES"], figsize=(8,4), color=["#2563eb","#dc2626"])
ax.set_title("STR cases and SLA breaches by priority"); ax.set_xlabel("priority"); ax.set_ylabel("count")
plt.tight_layout(); plt.show()
sla

## 6. Detection performance vs. synthetic ground truth
Scores the rule engine against the injected `IS_LAUNDERING` label (recall / precision / F1).

> **Optimistic by construction:** the synthetic laundering patterns are the ones the rules
> target, so this shows the rules catch their intended typologies — not production performance.

In [ ]:
perf = q('''
WITH labelled AS (
    SELECT t.TRANSACTION_KEY, s.IS_LAUNDERING,
           IFF(a.TRANSACTION_KEY IS NOT NULL, TRUE, FALSE) AS FLAGGED
    FROM CORE.FACT_TRANSACTIONS t
    JOIN STAGING.STG_TRANSACTIONS s ON s.TRANSACTION_ID = t.TRANSACTION_ID AND s.IS_VALID
    LEFT JOIN (SELECT DISTINCT TRANSACTION_KEY FROM CORE.FACT_AML_ALERTS) a
      ON a.TRANSACTION_KEY = t.TRANSACTION_KEY
), cm AS (
    SELECT SUM(IFF(IS_LAUNDERING AND FLAGGED,1,0)) TP,
           SUM(IFF(NOT IS_LAUNDERING AND FLAGGED,1,0)) FP,
           SUM(IFF(IS_LAUNDERING AND NOT FLAGGED,1,0)) FN,
           SUM(IFF(NOT IS_LAUNDERING AND NOT FLAGGED,1,0)) TN
    FROM labelled
)
SELECT TP, FP, FN, TN,
       ROUND(100.0*TP/NULLIF(TP+FN,0),1) AS RECALL_PCT,
       ROUND(100.0*TP/NULLIF(TP+FP,0),1) AS PRECISION_PCT,
       ROUND(2.0*TP/NULLIF(2*TP+FP+FN,0),3) AS F1
FROM cm''')
perf

## Findings

_(Fill in after running — e.g. which typologies dominate, escalation rate, STR conversion,
GGR trend, and the detection recall/precision/F1 above. These mirror the analyst version's
"Key Findings", now produced directly from the Snowflake reporting layer.)_